# Database and Big Data Pipeline — PT Kirimin
Kita membuat pipeline lokal dengan tahapan extract, validate, transform, dan load yang dapat diulang.

> Diverifikasi: Apache Airflow v3.3.2 — sumber: https://airflow.apache.org/docs/apache-airflow/3.3.2/ — tanggal cek: 2026-09-18
> Diverifikasi: Apache Spark v4.2.0 — sumber: https://spark.apache.org/releases/ — tanggal cek: 2026-09-18

In [ ]:
from dataclasses import dataclass
from datetime import date

@dataclass
class RunState:
    run_date: str
    status: str = 'created'
    rows_in: int = 0
    rows_out: int = 0

def run_partition(run_date, rows):
    state = RunState(run_date, 'running', len(rows), 0)
    valid = [row for row in rows if row.get('order_id') and row.get('order_value_idr', 0) >= 0]
    state.rows_out = len(valid)
    state.status = 'success'
    return state, valid

state, curated = run_partition('2026-09-18', [{'order_id':'O-1','order_value_idr':100}, {'order_id':None,'order_value_idr':50}])
state, curated

## Orchestration sketch
Dalam Airflow, fungsi seperti ini menjadi task dan dependency memastikan quality check selesai sebelum load. `run_date` membuat retry pada partition yang sama dapat diidentifikasi.

## Mini-exercise
1. Tambahkan checkpoint yang membuat rerun partition tidak menggandakan output.
2. Tambahkan status `quarantined` jika persentase invalid melewati threshold.
# TODO: implementasikan state transition Anda.

## Takeaway
Pipeline bukan hanya transformasi; ia juga state, dependency, retry, dan bukti hasil.